In [1]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [2]:
from fastapi import APIRouter
from services.mongodb import getDb
from services.redisclient import getRedis
from services.qdrantclient import getQdrantClient

In [3]:
router=APIRouter()

In [4]:
@router.get("/test")
async def test():
    deps={}
    try:
        db=getDb()
        await db.command("ping")
        deps["mongodb"]="ok"
    except Exception as e:
        deps["mongodb"]="error"
    r=getRedis()
    if r:
        try:
            await r.ping()
            deps["redis"]="ok"
        except Exception as e:
            deps["redis"]="error"
    else:
        deps["redis"]="unavailable"
    q=getQdrantClient()
    deps["qdrant"]="ok" if q else "unavailable"
    overall="ok" if deps.get("mongodb")=="ok" and deps.get("redis")=="ok" else "error"
    return {"status":overall,"dependencies":deps}